task 2                                                                                                                resturant recommendation                                                                                                  Objective: Create a restaurant recommendation system based on user preferences.                                      Steps: Preprocess the dataset by handling missing values and encoding categorical variables. Determine the criteria for restaurant recommendations (e.g., cuisine preference, price range). Implement a content-based filtering approach where users are recommended restaurants similar to their preferred criteria. Test the recommendation system by providing sample user preferences and evaluating the quality of recommendations

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder

In [2]:
data = pd.read_csv('resturant.csv', encoding='ISO-8859-1')

In [3]:
data.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  Switch to order menu 

In [5]:
data.describe()

,Restaurant ID,Country Code,Longitude,Latitude,Average Cost for two,Price range,Aggregate rating,Votes
count,9.551000e+03,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000
mean,9.051128e+06,18.365616,64.126574,25.854381,1199.210763,1.804837,2.666370,156.909748
std,8.791521e+06,56.750546,41.467058,11.007935,16121.183073,0.905609,1.516378,430.169145
min,5.300000e+01,1.000000,-157.948486,-41.330428,0.000000,1.000000,0.000000,0.000000
25%,3.019625e+05,1.000000,77.081343,28.478713,250.000000,1.000000,2.500000,5.000000
50%,6.004089e+06,1.000000,77.191964,28.570469,400.000000,2.000000,3.200000,31.000000
75%,1.835229e+07,1.000000,77.282006,28.642758,700.000000,2.000000,3.700000,131.000000
max,1.850065e+07,216.000000,174.832089,55.976980,800000.000000,4.000000,4.900000,10934.000000


In [6]:
data.isnull().sum()

Restaurant ID           0
Restaurant Name         0
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                9
Average Cost for two    0
Currency                0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Switch to order menu    0
Price range             0
Aggregate rating        0
Rating color            0
Rating text             0
Votes                   0
dtype: int64

In [7]:
data.dropna(inplace=True)

In [8]:
data.isnull().sum()

Restaurant ID           0
Restaurant Name         0
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                0
Average Cost for two    0
Currency                0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Switch to order menu    0
Price range             0
Aggregate rating        0
Rating color            0
Rating text             0
Votes                   0
dtype: int64

In [12]:
#create a refined dataframe
data = data[['Restaurant ID','Restaurant Name','Cuisines','Price range','Aggregate rating','Votes']]
data

,Restaurant ID,Restaurant Name,Cuisines,Price range,Aggregate rating,Votes
0,6317637,Le Petit Souffle,"French, Japanese, Desserts",3,4.8,314
1,6304287,Izakaya Kikufuji,Japanese,3,4.5,591
2,6300002,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",4,4.4,270
3,6318506,Ooma,"Japanese, Sushi",4,4.9,365
4,6314302,Sambo Kojin,"Japanese, Korean",4,4.8,229
...,...,...,...,...,...,...
9546,5915730,Naml?ñ Gurme,Turkish,3,4.1,788
9547,5908749,Ceviz A¨«¨«ac?ñ,"World Cuisine, Patisserie, Cafe",3,4.2,1034
9548,5915807,Huqqa,"Italian, World Cuisine",4,3.7,661
9549,5916112,A¨«¨«¨«k Kahve,Restaurant Cafe,4,4.0,901


In [13]:
data.duplicated().sum()

np.int64(0)

In [17]:
data['Restaurant Name'].duplicated().sum()

np.int64(2105)

In [18]:
data['Restaurant Name'].value_counts()

Restaurant Name
Cafe Coffee Day             83
Domino's Pizza              79
Subway                      63
Green Chick Chop            51
McDonald's                  48
                            ..
¨«ukura¨«¨«a Sofras?ñ     1
Gaga Manjero                 1
Cafemiz                      1
Nusr-Et                      1
Maori                        1
Name: count, Length: 7437, dtype: int64

In [20]:
#sorting the restaurants by name and rating
data = data.sort_values(by=['Restaurant Name','Aggregate rating'],ascending=False)
data.head()

,Restaurant ID,Restaurant Name,Cuisines,Price range,Aggregate rating,Votes
9523,6000871,¨«ukura¨«¨«a Sofras?ñ,"Kebab, Izgara",3,4.4,296
3120,18222559,{Niche} - Cafe & Bar,"North Indian, Chinese, Italian, Continental",3,4.1,492
9334,7100938,wagamama,"Japanese, Asian",4,3.7,131
9454,6401789,tashas,"Cafe, Mediterranean",4,4.1,374
4659,18361747,t Lounge by Dilmah,"Cafe, Tea, Desserts",2,3.6,34


In [21]:
data[data["Restaurant Name"]=="Cafe Coffee Day"].head()

,Restaurant ID,Restaurant Name,Cuisines,Price range,Aggregate rating,Votes
6430,5595,Cafe Coffee Day,Cafe,1,3.6,58
8432,594,Cafe Coffee Day,Cafe,1,3.6,125
3946,305736,Cafe Coffee Day,Cafe,1,3.5,35
5877,8828,Cafe Coffee Day,Cafe,1,3.5,50
3001,596,Cafe Coffee Day,Cafe,1,3.4,277


In [22]:
data.drop_duplicates(subset='Restaurant Name',keep='first',inplace=True)    

In [23]:
data

,Restaurant ID,Restaurant Name,Cuisines,Price range,Aggregate rating,Votes
9523,6000871,¨«ukura¨«¨«a Sofras?ñ,"Kebab, Izgara",3,4.4,296
3120,18222559,{Niche} - Cafe & Bar,"North Indian, Chinese, Italian, Continental",3,4.1,492
9334,7100938,wagamama,"Japanese, Asian",4,3.7,131
9454,6401789,tashas,"Cafe, Mediterranean",4,4.1,374
4659,18361747,t Lounge by Dilmah,"Cafe, Tea, Desserts",2,3.6,34
...,...,...,...,...,...,...
8692,18317511,#Urban Caf¨«¨«,"North Indian, Chinese, Italian",2,3.3,49
6998,18336489,#OFF Campus,"Cafe, Continental, Italian, Fast Food",2,3.7,216
2613,18311951,#InstaFreeze,Ice Cream,1,0.0,2
9148,18378803,#Dilliwaala6,North Indian,3,3.7,124


In [25]:
data.shape

(7437, 6)

In [26]:
data['Restaurant Name'].value_counts()

Restaurant Name
#45                         1
¨«ukura¨«¨«a Sofras?ñ    1
{Niche} - Cafe & Bar        1
wagamama                    1
tashas                      1
                           ..
Zune - Piccadily Hotel      1
Zunzi's                     1
Zustt Yummy                 1
Zync - Rosewood Hotel       1
bu¨«¨«no                  1
Name: count, Length: 7437, dtype: int64

In [27]:
data = data[data['Aggregate rating']>3.9]
data

,Restaurant ID,Restaurant Name,Cuisines,Price range,Aggregate rating,Votes
9523,6000871,¨«ukura¨«¨«a Sofras?ñ,"Kebab, Izgara",3,4.4,296
3120,18222559,{Niche} - Cafe & Bar,"North Indian, Chinese, Italian, Continental",3,4.1,492
9454,6401789,tashas,"Cafe, Mediterranean",4,4.1,374
9385,6113857,sketch Gallery,"British, Contemporary",4,4.5,148
1837,18418247,feel ALIVE,"North Indian, American, Asian, Biryani",3,4.7,69
...,...,...,...,...,...,...
1468,18408054,19 Flavours Biryani,"Mughlai, Hyderabadi",2,4.1,84
2484,18233317,145 Kala Ghoda,"Fast Food, Beverages, Desserts",3,4.2,1606
2292,2100784,11th Avenue Cafe Bistro,"Cafe, American, Italian, Continental",2,4.1,377
751,2600031,10 Downing Street,"North Indian, Chinese",3,4.0,257


In [28]:
#splitting cuisines into list
data['Cuisines'] = data['Cuisines'].str.split(', ')
data

C:\Users\EMC\AppData\Local\Temp\ipykernel_1924\3954604057.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Cuisines'] = data['Cuisines'].str.split(', ')


,Restaurant ID,Restaurant Name,Cuisines,Price range,Aggregate rating,Votes
9523,6000871,¨«ukura¨«¨«a Sofras?ñ,"[Kebab, Izgara]",3,4.4,296
3120,18222559,{Niche} - Cafe & Bar,"[North Indian, Chinese, Italian, Continental]",3,4.1,492
9454,6401789,tashas,"[Cafe, Mediterranean]",4,4.1,374
9385,6113857,sketch Gallery,"[British, Contemporary]",4,4.5,148
1837,18418247,feel ALIVE,"[North Indian, American, Asian, Biryani]",3,4.7,69
...,...,...,...,...,...,...
1468,18408054,19 Flavours Biryani,"[Mughlai, Hyderabadi]",2,4.1,84
2484,18233317,145 Kala Ghoda,"[Fast Food, Beverages, Desserts]",3,4.2,1606
2292,2100784,11th Avenue Cafe Bistro,"[Cafe, American, Italian, Continental]",2,4.1,377
751,2600031,10 Downing Street,"[North Indian, Chinese]",3,4.0,257


In [29]:
data = data.explode('Cuisines')
data

,Restaurant ID,Restaurant Name,Cuisines,Price range,Aggregate rating,Votes
9523,6000871,¨«ukura¨«¨«a Sofras?ñ,Kebab,3,4.4,296
9523,6000871,¨«ukura¨«¨«a Sofras?ñ,Izgara,3,4.4,296
3120,18222559,{Niche} - Cafe & Bar,North Indian,3,4.1,492
3120,18222559,{Niche} - Cafe & Bar,Chinese,3,4.1,492
3120,18222559,{Niche} - Cafe & Bar,Italian,3,4.1,492
...,...,...,...,...,...,...
2292,2100784,11th Avenue Cafe Bistro,Italian,2,4.1,377
2292,2100784,11th Avenue Cafe Bistro,Continental,2,4.1,377
751,2600031,10 Downing Street,North Indian,3,4.0,257
751,2600031,10 Downing Street,Chinese,3,4.0,257


In [30]:
data['Cuisines'].value_counts()

Cuisines
North Indian      270
Italian           237
Chinese           200
Continental       199
Cafe              177
                 ... 
Fish and Chips      1
Cuban               1
Mangalorean         1
New American        1
Bubble Tea          1
Name: count, Length: 128, dtype: int64

In [31]:
data['Cuisines'].nunique()

128

In [32]:
data['Cuisines'].unique()

array(['Kebab', 'Izgara', 'North Indian', 'Chinese', 'Italian',
       'Continental', 'Cafe', 'Mediterranean', 'British', 'Contemporary',
       'American', 'Asian', 'Biryani', 'International', 'Sandwich',
       'Desserts', 'Burger', 'Vegetarian', 'Bakery', 'Pizza', 'Juices',
       'Beverages', 'Healthy Food', 'Thai', 'Turkish Pizza', 'Japanese',
       'Sushi', 'Ramen', 'Steak', 'Brazilian', 'Pakistani',
       'South Indian', 'Seafood', 'Kerala', 'Mughlai', 'Indian', 'BBQ',
       'Dim Sum', 'Street Food', 'D\x8b¨«_ner', 'European', 'Fast Food',
       'Breakfast', 'Hawaiian', 'Lebanese', 'Latin American', 'Argentine',
       'Filipino', 'Bar Food', 'Restaurant Cafe', 'Tapas', 'Finger Food',
       'Sri Lankan', 'Andhra', 'Chettinad', 'Bengali', 'Western',
       'Rajasthani', 'Vietnamese', 'Coffee and Tea', 'Indonesian',
       'Mexican', 'Modern Indian', 'Afghani', 'Hyderabadi', 'Cajun',
       'Korean', 'Arabian', 'Scottish', 'Malaysian', 'Pub Food',
       'Southern', 'Spanish'

In [34]:
restoXcuisines = pd.crosstab(data['Restaurant Name'], data['Cuisines'])
restoXcuisines

Cuisines,Afghani,African,American,Andhra,Arabian,Argentine,Asian,Asian Fusion,Australian,Awadhi,...,Teriyaki,Tex-Mex,Thai,Tibetan,Turkish,Turkish Pizza,Vegetarian,Vietnamese,Western,World Cuisine
Restaurant Name,,,,,,,,,,,,,,,,,,,,,
'Ohana,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10 Downing Street,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
11th Avenue Cafe Bistro,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
145 Kala Ghoda,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
19 Flavours Biryani,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
feel ALIVE,0,0,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
sketch Gallery,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
tashas,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [37]:
data['Restaurant Name'].sample(20, random_state=194)

7729                           Indian Saffron Co.
3013                           Naturals Ice Cream
422         Sansei Seafood Restaurant & Sushi Bar
81                       Sainte Marie Gastronomia
3310                                Spezia Bistro
7849                              Cafeteria & Co.
2407                             India Restaurant
637                               Sheroes Hangout
1858                           Boombox Brewstreet
2393    Aangan - Downtown Multicuisine Restaurant
9339                                         Bank
7067                                     Pa Pa Ya
728                                          Toit
6447                                 Bakerz Lodge
9469                           The Belgian Triple
7727                                 Cafe Connect
9532                               Masaba¨«¨«?ñ
734                            ECHOES Koramangala
3703                   Sakley's The Mountain Cafe
6714                                  Showstopper


In [65]:
# Calculate similarity
similarity_matrix = cosine_similarity(cuisine_encoded)


In [42]:
# Convert into a DataFrame
similarity_matrix

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(9542, 9542))

In [51]:
# Measure Similarity
from sklearn.metrics import jaccard_score

In [53]:
jaccard_score(restoXcuisines.loc["Olive Bistro"].values, restoXcuisines.loc["Rose Cafe"].values)

np.float64(0.3333333333333333)

In [56]:
# Create Similarity Value Dataframe
from scipy.spatial.distance import pdist, squareform
jaccardDist = pdist(restoXcuisines.values, metric='jaccard')
jaccardMatrix = squareform(jaccardDist)
jaccardSim = 1 - jaccardMatrix
dataJaccard = pd.DataFrame(jaccardSim,index=restoXcuisines.index,columns=restoXcuisines.index)

dataJaccard

Restaurant Name,'Ohana,10 Downing Street,11th Avenue Cafe Bistro,145 Kala Ghoda,19 Flavours Biryani,1918 Bistro & Grill,2 Dog,22nd Parallel,3 Wise Monkeys,38 Barracks,...,Zoeys Pizzeria,Zolocrust - Hotel Clarks Amer,Zombie Burger + Drink Lab,Zuka Choco-la,Zunzi's,feel ALIVE,sketch Gallery,tashas,{Niche} - Cafe & Bar,¨«ukura¨«¨«a Sofras?ñ
Restaurant Name,,,,,,,,,,,,,,,,,,,,,
'Ohana,1.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.00,0.000000,0.0,0.0,0.000000,0.0
10 Downing Street,0.0,1.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.200000,...,0.0,0.0,0.0,0.000000,0.00,0.200000,0.0,0.0,0.500000,0.0
11th Avenue Cafe Bistro,0.0,0.0,1.000000,0.0,0.0,0.0,0.166667,0.0,0.0,0.333333,...,0.0,0.4,0.0,0.000000,0.00,0.142857,0.0,0.2,0.333333,0.0
145 Kala Ghoda,0.0,0.0,0.000000,1.0,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.2,0.333333,0.00,0.000000,0.0,0.0,0.000000,0.0
19 Flavours Biryani,0.0,0.0,0.000000,0.0,1.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.00,0.000000,0.0,0.0,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
feel ALIVE,0.0,0.2,0.142857,0.0,0.0,0.0,0.166667,0.0,0.0,0.600000,...,0.0,0.0,0.0,0.000000,0.00,1.000000,0.0,0.0,0.142857,0.0
sketch Gallery,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.00,0.000000,1.0,0.0,0.000000,0.0
tashas,0.0,0.0,0.200000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.25,0.000000,0.0,1.0,0.000000,0.0


In [57]:
#restaurant NAME SAMPLE
data['Restaurant Name'].sample(20, random_state=194)

7729                           Indian Saffron Co.
3013                           Naturals Ice Cream
422         Sansei Seafood Restaurant & Sushi Bar
81                       Sainte Marie Gastronomia
3310                                Spezia Bistro
7849                              Cafeteria & Co.
2407                             India Restaurant
637                               Sheroes Hangout
1858                           Boombox Brewstreet
2393    Aangan - Downtown Multicuisine Restaurant
9339                                         Bank
7067                                     Pa Pa Ya
728                                          Toit
6447                                 Bakerz Lodge
9469                           The Belgian Triple
7727                                 Cafe Connect
9532                               Masaba¨«¨«?ñ
734                            ECHOES Koramangala
3703                   Sakley's The Mountain Cafe
6714                                  Showstopper


In [64]:
# Make Recommendation

# Input Initial Restaurant Name
resto = 'Ooma'

sim = dataJaccard.loc[resto].sort_values(ascending=False)

sim = pd.DataFrame({'restaurant_name': sim.index, 'simScore': sim.values})
sim = sim[(sim['restaurant_name']!= resto) & (sim['simScore']>=0.7)].head(5)

# Merge The Rating

RestoRec = pd.merge(sim, data[['Restaurant Name', 'Aggregate rating']], how='inner', left_on='restaurant_name', right_on='Restaurant Name')
FinalRestoRec = RestoRec.sort_values('Aggregate rating', ascending=False).drop_duplicates('restaurant_name', keep='first')
FinalRestoRec

,restaurant_name,simScore,Restaurant Name,Aggregate rating
3,Sushi Masa,1.0,Sushi Masa,4.9
7,Miyabi 9,1.0,Miyabi 9,4.8
8,Nagai,1.0,Nagai,4.3
1,Osaka,1.0,Osaka,4.2
5,Guppy,1.0,Guppy,4.1
